<a href="https://colab.research.google.com/github/erickvaldezsallagos/SISTEMAS-FINANCIEROS-Y-COMERCIALES/blob/main/Tarea_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SISTEMAS FINANCIEROS Y COMERCIALES** \
**EQUIPO 3**\
TAREA A: Realizar un programa consultando una base de datos. Calcule SMA, WMA y EMA de un menú de acciones

**Este programa consulta datos históricos de una fuente de datos financiera (Yahoo Finance) y calcula tres tipos de medias móviles para un menú de acciones predefinido.**

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf


def calcular_medias_moviles(
    ticker: str, start_date: str, end_date: str, window: int = 20
):
  """Descarga datos históricos de una acción y calcula SMA, WMA y EMA con manejo de errores."""
  print(
      f"\n--- Descargando datos para {ticker} desde {start_date} hasta"
      f" {end_date} ---"
  )

  try:
    # Descarga de datos con yfinance
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)

    if df.empty:
      print(
          f"[Aviso] No se encontraron datos para el ticker '{ticker}'. Verifica"
          " la clave."
      )
      return None

    if isinstance(df.columns, pd.MultiIndex):
      if ('Close', ticker) in df.columns:
        close_prices = df[('Close', ticker)]
      else:
        close_prices = df['Close'].iloc[:, 0]
    else:
      close_prices = (
          df['Close'] if 'Close' in df.columns else df.iloc[:, 0]
      )

    close_prices = pd.Series(close_prices).dropna()

    if len(close_prices) < window:
      print(
          f"[Aviso] Registros insuficientes ({len(close_prices)}) para la"
          f" ventana de {window} periodos en {ticker}."
      )
      return None


    resultado = pd.DataFrame(index=close_prices.index)
    resultado['Close'] = close_prices

    # 1. Media Móvil Simple (SMA)
    resultado['SMA'] = resultado['Close'].rolling(window=window).mean()

    # 2. Media Móvil Ponderada (WMA)
    weights = np.arange(1, window + 1)
    resultado['WMA'] = (
        resultado['Close']
        .rolling(window=window)
        .apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
    )

    # 3. Media Móvil Exponencial (EMA)
    resultado['EMA'] = resultado['Close'].ewm(span=window, adjust=False).mean()

    return resultado

  except Exception as e:
    print(f"[Error] Ocurrió un error al procesar el ticker {ticker}: {e}")
    return None


# MENÚ DE ACCIONES Y EJECUCIÓN
if __name__ == '__main__':
  start_date = '2021-12-13'
  end_date = '2022-12-13'
  periodo_ma = 20  # Número de periodos (N)


  menu_acciones = ['GOOGL', 'AAPL', 'MSFT', 'CX']

  for clave in menu_acciones:
    datos_calculados = calcular_medias_moviles(
        clave, start_date, end_date, window=periodo_ma
    )

    if datos_calculados is not None:
      print(f'\nResultados recientes para: {clave}')
      print(
          datos_calculados[['Close', 'SMA', 'WMA', 'EMA']]
          .tail(5)
          .to_string()
      )
      print('=' * 60)


--- Descargando datos para GOOGL desde 2021-12-13 hasta 2022-12-13 ---

Resultados recientes para: GOOGL
                Close        SMA        WMA        EMA
Date                                                  
2022-12-06  96.065865  95.790485  97.032617  96.371607
2022-12-07  94.045113  96.089639  96.866391  96.150036
2022-12-08  92.826683  96.406128  96.555633  95.833526
2022-12-09  91.955002  96.351151  96.131716  95.464143
2022-12-12  92.430458  96.197612  95.758317  95.175221

--- Descargando datos para AAPL desde 2021-12-13 hasta 2022-12-13 ---

Resultados recientes para: AAPL
                 Close         SMA         WMA         EMA
Date                                                      
2022-12-06  140.403900  144.250731  144.565658  144.026208
2022-12-07  138.468460  144.321469  144.014966  143.496898
2022-12-08  140.148468  144.703649  143.617537  143.178000
2022-12-09  139.667053  144.472279  143.137861  142.843624
2022-12-12  141.956177  144.216345  142.898233  142

/tmp/ipykernel_1328/3097089460.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1328/3097089460.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1328/3097089460.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1328/3097089460.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)
